[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Requirements and Pinning &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folder, `run` and `pip`, and imports `SpecifierSet` and
`Version`. Run it first, then the tasks in order, since tasks 2, 5 and 6 use what tasks 1 and 2
made. The last cell removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

from packaging.specifiers import SpecifierSet
from packaging.version import Version

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide


def run(*command, folder=PROJECT):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments, folder=PROJECT):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", "--no-color",
               *arguments, folder=folder)


print("ready:", PROJECT)


ready: scratch/stations


**1.** An environment, frozen into a file.


In [2]:
run(sys.executable, "-m", "venv", "--without-pip", "task-env")
code, printed = pip("task-env", "install", "-q", "packaging==21.3", "pyparsing==3.3.2")
print("install exit code:", code)

code, frozen = pip("task-env", "freeze")
(PROJECT / "task-requirements.txt").write_text(frozen + "\n")
print((PROJECT / "task-requirements.txt").read_text(), end="")


install exit code: 0
packaging==21.3
pyparsing==3.3.2


`freeze` prints the requirements without a final newline, and a text file ends with one, which the
`+ "\n"` adds.


**2.** Rebuilt from the file.


In [3]:
run(sys.executable, "-m", "venv", "--without-pip", "task-copy")
code, printed = pip("task-copy", "install", "-q", "-r", "task-requirements.txt")
print("install exit code:", code)

print("the same packages:", pip("task-copy", "freeze")[1] == frozen)


install exit code: 0
the same packages: True


Both environments hold the same two packages at the same versions, which is what a pinned file
promises.


**3.** Two compatible releases.


In [4]:
releases = ["1.0", "1.4", "1.4.5", "1.5", "2.0"]

for specifier in ["~=1.4", "~=1.4.0"]:
    print(f"{specifier:<8} {list(SpecifierSet(specifier).filter(releases))}")


~=1.4    ['1.4', '1.4.5', '1.5']
~=1.4.0  ['1.4', '1.4.5']


`~=1.4` means 1.4 or later within 1, so 1.5 is in and 2.0 is not. `~=1.4.0` means 1.4.0 or later
within 1.4, so 1.5 is out, and 1.4 is in, since 1.4 and 1.4.0 are the same version.


**4.** Sorted as text, and as versions.


In [5]:
tags = ["1.10", "1.9", "1.10.post1", "1.10rc1"]

print("as text:    ", sorted(tags))
print("as versions:", sorted(tags, key=Version))


as text:     ['1.10', '1.10.post1', '1.10rc1', '1.9']
as versions: ['1.9', '1.10rc1', '1.10', '1.10.post1']


As versions, the release candidate comes before 1.10 and the post-release after it, and 1.9 comes
first, where text puts it last.


**5.** A constraints file, tried with --dry-run.


In [6]:
(PROJECT / "task-constraints.txt").write_text("pyparsing==3.2.0\n")
run(sys.executable, "-m", "venv", "--without-pip", "task-empty")

code, printed = pip("task-empty", "install", "--dry-run", "packaging==21.3", "-c", "task-constraints.txt")
print(next(line for line in printed.splitlines() if line.startswith("Would install")))


Would install packaging-21.3 pyparsing-3.2.0


The dry run went into a new, empty environment, so that the line shows both packages. In `task-env`,
which already holds `packaging` 21.3, it would show only the change to `pyparsing`.


**6.** A development file that includes the first one.


In [7]:
(PROJECT / "task-dev.txt").write_text("-r task-requirements.txt\npytest==8.4.2\n")

code, printed = pip("task-empty", "install", "--dry-run", "-r", "task-dev.txt")
would_install = next(line for line in printed.splitlines() if line.startswith("Would install"))
print(sorted((item.rsplit("-", 1)[0] for item in would_install.split()[2:]), key=str.lower))


['iniconfig', 'packaging', 'pluggy', 'Pygments', 'pyparsing', 'pytest']


The two packages of `task-requirements.txt`, pytest, and the packages pytest needs. Each item in the
`Would install` line is a name and a version joined by the last `-`, which `rsplit("-", 1)` splits
at.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Requirements and Pinning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/08-requirements-and-pinning.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
